# PDF Extraction with AI_EXTRACT + responseFormat Schema

This notebook extracts structured data from PDFs using `AI_EXTRACT` with typed `responseFormat` schemas. The schema approach returns **parallel arrays** (one per column) that LATERAL FLATTEN unpacks into rows.

**Document types:**
1. **Bank Statements** — transactions with date, description, amount, balance
2. **Construction Spec Sheets** — equipment schedules with tag, service, CFM, ESP, etc.

**Key technique:** The `responseFormat` schema tells the model exactly what structure to return, eliminating token truncation issues and producing clean columnar output.

### Setup — Database, Warehouse, and Stages

Creates the database, warehouse, and two internal stages with directory tables enabled for file listing.

In [ ]:
%%sql -r context_result
USE ROLE SYSADMIN;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

CREATE DATABASE IF NOT EXISTS AI_EXTRACT_DB;

USE WAREHOUSE COMPUTE_WH;
USE DATABASE AI_EXTRACT_DB;
USE SCHEMA PUBLIC;

### Create Stages

Creates two internal stages for bank statements and construction docs.

In [ ]:
%%sql -r stage_result
CREATE OR REPLACE STAGE bank_statements_stage
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

CREATE OR REPLACE STAGE construction_specs_stage
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

## Upload PDFs

Upload your PDF files to the appropriate stage via Snowsight:
- **Bank statements** → `@bank_statements_stage`
- **Construction specs** → `@construction_specs_stage`

Or via SQL: `PUT file:///path/to/file.pdf @stage_name;`

### Verify PDFs on Stage

Lists files on both stages to confirm PDFs were uploaded successfully.

In [ ]:
%%sql -r stage_files
LIST @bank_statements_stage;
LIST @construction_specs_stage;

---
## Bank Statements — Extract with responseFormat Schema

Extracts metadata (issuer, recipient, account number, statement period) and all transactions as parallel arrays. The `column_ordering` key ensures consistent output structure.

In [ ]:
SELECT
    RELATIVE_PATH,
    AI_EXTRACT(
        file => TO_FILE('@bank_statements_stage', RELATIVE_PATH),
        responseFormat => {
            'schema': {
                'type': 'object',
                'properties': {
                    'document_type': { 'type': 'string', 'description': 'Type of document (e.g. Bank Statement, Credit Card Statement)' },
                    'issuer':        { 'type': 'string', 'description': 'Bank or financial institution name' },
                    'recipient':     { 'type': 'string', 'description': 'Account holder name' },
                    'account_number': { 'type': 'string', 'description': 'Masked account number' },
                    'statement_period': { 'type': 'string', 'description': 'Statement date range' },
                    'transactions': {
                        'type': 'object',
                        'description': 'All transactions in the statement.',
                        'column_ordering': ['date', 'description', 'amount', 'balance'],
                        'properties': {
                            'date':        { 'description': 'Transaction date', 'type': 'array' },
                            'description': { 'description': 'Transaction description', 'type': 'array' },
                            'amount':      { 'description': 'Amount, negative for debits/withdrawals, number only', 'type': 'array' },
                            'balance':     { 'description': 'Running balance after transaction, number only', 'type': 'array' }
                        }
                    }
                }
            }
        }
    ) AS extraction_json
FROM DIRECTORY(@bank_statements_stage)
WHERE RELATIVE_PATH LIKE '%.pdf';

### Bank Statements — LATERAL FLATTEN + Edge Case Normalization

Flattens the parallel arrays into one row per transaction. Handles:
- `$` and `,` in amounts/balances
- `+` prefix on credits
- `—` (em-dash) on Balance Forward rows
- `None` values filtered out (credit card statements with no running balance)

In [ ]:
WITH extracted AS (
    SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
),
flattened AS (
    SELECT
        e.RELATIVE_PATH,
        e.EXTRACTION_JSON:response:document_type::STRING AS document_type,
        e.EXTRACTION_JSON:response:issuer::STRING AS issuer,
        e.EXTRACTION_JSON:response:recipient::STRING AS recipient,
        e.EXTRACTION_JSON:response:account_number::STRING AS account_number,
        e.EXTRACTION_JSON:response:statement_period::STRING AS statement_period,
        idx.INDEX + 1 AS line_number,
        e.EXTRACTION_JSON:response:transactions:date[idx.INDEX]::STRING AS transaction_date,
        e.EXTRACTION_JSON:response:transactions:description[idx.INDEX]::STRING AS description,
        e.EXTRACTION_JSON:response:transactions:amount[idx.INDEX]::STRING AS amount_raw,
        e.EXTRACTION_JSON:response:transactions:balance[idx.INDEX]::STRING AS balance_raw
    FROM extracted e,
        LATERAL FLATTEN(input => e.EXTRACTION_JSON:response:transactions:date) idx
)
SELECT
    RELATIVE_PATH,
    document_type,
    issuer,
    recipient,
    account_number,
    statement_period,
    line_number,
    transaction_date,
    description,
    TRY_TO_DOUBLE(
        REPLACE(REPLACE(REPLACE(REPLACE(amount_raw, '$', ''), ',', ''), '+', ''), '—', '')
    ) AS amount,
    TRY_TO_DOUBLE(
        REPLACE(REPLACE(REPLACE(balance_raw, '$', ''), ',', ''), '+', '')
    ) AS balance
FROM flattened
WHERE amount_raw IS NOT NULL
  AND amount_raw NOT IN ('None', 'null', '')
ORDER BY RELATIVE_PATH, line_number;

---
## Construction Spec Sheets — Extraction Pipeline

Same `responseFormat` schema pattern. The schema includes columns for all document types (AHU, ductwork, controls, chiller, exhaust) — non-applicable fields return NULL via `NULLIF(..., 'None')`.

In [ ]:
SELECT
    RELATIVE_PATH,
    AI_EXTRACT(
        file => TO_FILE('@construction_specs_stage', RELATIVE_PATH),
        responseFormat => {
            'schema': {
                'type': 'object',
                'properties': {
                    'document_number':   { 'type': 'string', 'description': 'Document number (e.g. HVAC-001)' },
                    'revision':          { 'type': 'string', 'description': 'Revision letter (e.g. A, B, C)' },
                    'date':              { 'type': 'string', 'description': 'Document date in YYYY-MM-DD' },
                    'project_name':      { 'type': 'string', 'description': 'Project name' },
                    'issuing_firm':      { 'type': 'string', 'description': 'Engineering firm name' },
                    'section_number':    { 'type': 'string', 'description': 'CSI section number (e.g. 23 73 13)' },
                    'section_title':     { 'type': 'string', 'description': 'Section title' },
                    'scope_of_work':     { 'type': 'string', 'description': 'Scope of work summary' },
                    'equipment_schedule': {
                        'type': 'object',
                        'description': 'Equipment schedule table. Extract ALL columns from the table.',
                        'column_ordering': ['tag', 'service', 'cfm_design', 'esp_in_wg', 'cooling_mbh', 'heating_mbh', 'oa_pct', 'pressure_class', 'material', 'gauge', 'joints', 'insulation', 'controller_type', 'io_points', 'protocol', 'quantity', 'capacity_tons', 'motor_hp', 'drive_type'],
                        'properties': {
                            'tag':             { 'description': 'Equipment tag or duct system name', 'type': 'array' },
                            'service':         { 'description': 'Area or system served', 'type': 'array' },
                            'cfm_design':      { 'description': 'CFM at design, number only. None if not applicable.', 'type': 'array' },
                            'esp_in_wg':       { 'description': 'Static pressure in inches w.g., number only. None if not applicable.', 'type': 'array' },
                            'cooling_mbh':     { 'description': 'Cooling capacity in MBH, number only. None if not applicable.', 'type': 'array' },
                            'heating_mbh':     { 'description': 'Heating capacity in MBH, number only. None if not applicable.', 'type': 'array' },
                            'oa_pct':          { 'description': 'Outside air percentage, number only. None if not applicable.', 'type': 'array' },
                            'pressure_class':  { 'description': 'Duct pressure class (e.g. 2-in. SP). None if not applicable.', 'type': 'array' },
                            'material':        { 'description': 'Material (e.g. G90 Galv. Steel). None if not applicable.', 'type': 'array' },
                            'gauge':           { 'description': 'Metal gauge (e.g. 20 ga.). None if not applicable.', 'type': 'array' },
                            'joints':          { 'description': 'Joint type (e.g. TDC/TDF, S&D). None if not applicable.', 'type': 'array' },
                            'insulation':      { 'description': 'Insulation spec. None if not applicable.', 'type': 'array' },
                            'controller_type': { 'description': 'DDC controller type. None if not applicable.', 'type': 'array' },
                            'io_points':       { 'description': 'Number of I/O points. None if not applicable.', 'type': 'array' },
                            'protocol':        { 'description': 'Communication protocol (e.g. BACnet). None if not applicable.', 'type': 'array' },
                            'quantity':        { 'description': 'Quantity of units. None if not applicable.', 'type': 'array' },
                            'capacity_tons':   { 'description': 'Capacity in tons (chillers/cooling towers). None if not applicable.', 'type': 'array' },
                            'motor_hp':        { 'description': 'Motor horsepower. None if not applicable.', 'type': 'array' },
                            'drive_type':      { 'description': 'Drive type (e.g. VFD, Direct). None if not applicable.', 'type': 'array' }
                        }
                    }
                }
            }
        }
    ) AS extraction_json
FROM DIRECTORY(@construction_specs_stage)
WHERE RELATIVE_PATH LIKE '%.pdf';

### Construction Specs — Extract Metadata + Equipment Schedule

Extracts document metadata and equipment schedule as parallel arrays. The schema covers all HVAC document types (AHU performance, ductwork specs, DDC controllers, chiller plant, exhaust fans) in a single unified schema.

In [ ]:
WITH extracted AS (
    SELECT
        "RELATIVE_PATH" AS RELATIVE_PATH,
        "EXTRACTION_JSON" AS EXTRACTION_JSON
    FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
),
flattened AS (
    SELECT
        e.RELATIVE_PATH,
        e.EXTRACTION_JSON:response:document_number::STRING AS document_number,
        e.EXTRACTION_JSON:response:revision::STRING AS revision,
        e.EXTRACTION_JSON:response:date::STRING AS doc_date,
        e.EXTRACTION_JSON:response:project_name::STRING AS project_name,
        e.EXTRACTION_JSON:response:issuing_firm::STRING AS issuing_firm,
        e.EXTRACTION_JSON:response:section_number::STRING AS section_number,
        e.EXTRACTION_JSON:response:section_title::STRING AS section_title,
        e.EXTRACTION_JSON:response:scope_of_work::STRING AS scope_of_work,
        idx.INDEX + 1 AS line_number,
        e.EXTRACTION_JSON:response:equipment_schedule:tag[idx.INDEX]::STRING AS tag,
        e.EXTRACTION_JSON:response:equipment_schedule:service[idx.INDEX]::STRING AS service,
        TRY_TO_DOUBLE(REPLACE(e.EXTRACTION_JSON:response:equipment_schedule:cfm_design[idx.INDEX]::STRING, ',', '')) AS cfm_design,
        TRY_TO_DOUBLE(e.EXTRACTION_JSON:response:equipment_schedule:esp_in_wg[idx.INDEX]::STRING) AS esp_in_wg,
        TRY_TO_DOUBLE(REPLACE(e.EXTRACTION_JSON:response:equipment_schedule:cooling_mbh[idx.INDEX]::STRING, ',', '')) AS cooling_mbh,
        TRY_TO_DOUBLE(REPLACE(e.EXTRACTION_JSON:response:equipment_schedule:heating_mbh[idx.INDEX]::STRING, ',', '')) AS heating_mbh,
        TRY_TO_DOUBLE(REPLACE(e.EXTRACTION_JSON:response:equipment_schedule:oa_pct[idx.INDEX]::STRING, '%', '')) AS oa_pct,
        NULLIF(e.EXTRACTION_JSON:response:equipment_schedule:pressure_class[idx.INDEX]::STRING, 'None') AS pressure_class,
        NULLIF(e.EXTRACTION_JSON:response:equipment_schedule:material[idx.INDEX]::STRING, 'None') AS material,
        NULLIF(e.EXTRACTION_JSON:response:equipment_schedule:gauge[idx.INDEX]::STRING, 'None') AS gauge,
        NULLIF(e.EXTRACTION_JSON:response:equipment_schedule:joints[idx.INDEX]::STRING, 'None') AS joints,
        NULLIF(e.EXTRACTION_JSON:response:equipment_schedule:insulation[idx.INDEX]::STRING, 'None') AS insulation,
        NULLIF(e.EXTRACTION_JSON:response:equipment_schedule:controller_type[idx.INDEX]::STRING, 'None') AS controller_type,
        NULLIF(e.EXTRACTION_JSON:response:equipment_schedule:io_points[idx.INDEX]::STRING, 'None') AS io_points,
        NULLIF(e.EXTRACTION_JSON:response:equipment_schedule:protocol[idx.INDEX]::STRING, 'None') AS protocol,
        NULLIF(e.EXTRACTION_JSON:response:equipment_schedule:quantity[idx.INDEX]::STRING, 'None') AS quantity,
        TRY_TO_DOUBLE(REPLACE(NULLIF(e.EXTRACTION_JSON:response:equipment_schedule:capacity_tons[idx.INDEX]::STRING, 'None'), ',', '')) AS capacity_tons,
        TRY_TO_DOUBLE(REPLACE(NULLIF(e.EXTRACTION_JSON:response:equipment_schedule:motor_hp[idx.INDEX]::STRING, 'None'), ',', '')) AS motor_hp,
        NULLIF(e.EXTRACTION_JSON:response:equipment_schedule:drive_type[idx.INDEX]::STRING, 'None') AS drive_type
    FROM extracted e,
        LATERAL FLATTEN(input => e.EXTRACTION_JSON:response:equipment_schedule:tag) idx
)
SELECT *
FROM flattened
WHERE tag IS NOT NULL AND tag != 'None'
ORDER BY RELATIVE_PATH, line_number;

### Construction Specs — LATERAL FLATTEN into Rows

Flattens equipment schedule arrays into one row per equipment item. Uses `NULLIF(..., 'None')` to convert non-applicable fields to proper SQL NULLs. Numeric columns parsed with `TRY_TO_DOUBLE` + `REPLACE` for commas and `%` signs.

## Cleanup

Run the cell below to remove all resources created during this demo.

In [ ]:
%%sql -r cleanup_result
DROP TABLE IF EXISTS bank_statement_extracts;
DROP TABLE IF EXISTS construction_spec_extracts;
DROP STAGE IF EXISTS bank_statements_stage;
DROP STAGE IF EXISTS construction_specs_stage;

## Persist Extracted Data to Tables

Store each document type in its own table.